In [ ]:
library(tidyverse)
library(ggplot2)
library(scales)
library(dplyr)
library(fixest)
library(tidyverse)
library(AER)
library(stargazer)
data <- read_csv("../data/final_hcris_data.csv")

how many hospitals filed more than one report in the same year? Show answer as a line graph of the number of hospitals over time.


In [ ]:
report_data <- data %>%
    filter(year >= 2009 & year <= 2019) %>%
    filter(source != "unique reports") %>%
    group_by(year) %>%
    summarize(count = n_distinct(provider_number))

# line plot
ggplot(report_data, aes(x= year, y=count)) +
    geom_line() +
    geom_point() +
    scale_x_continuous(breaks = 2009:2019) + 
    labs(
        title = "Hospitals with more than One Report",
        x = "Year",
        y = "Number of Hospitals"
    ) + 
        theme_minimal()

# Question 2
after removing/combining multiple reports, how many unique hospital IDs (Medicare provider numbers) exist in the data?


In [ ]:
unique_hospitals <- data %>%
    filter(year >= 2009 & year <= 2019) %>%
    summarize(unique_count = n_distinct(provider_number))

print(paste("Total unique hospitas IDs", unique_hospitals$unique_count))

In [ ]:
unique_hospitalsyr <- data %>%
    filter(year >= 2009 & year <= 2019) %>%
    group_by(year) %>%
    summarize(hospitals = n_distinct(provider_number))

unique_hospitalsyr

# Question 3
What is the distribution of total charges (tot_charges) in each year? Show results in a violin plot with charges on the y-axis and years on the x-axis


In [ ]:
data_filter <- data %>%
    filter(year >= 2009 & year <= 2019) %>%
    filter(!is.na(tot_charges), tot_charges >0) 

# violin plot
ggplot(data_filter, aes(x = factor(year), y = tot_charges)) +
    geom_violin() +
    
    labs(
        title = "Distribution of total hospital charges",
        x = "fiscal year",
        y = "total charges ($)"
    ) +
    theme_minimal()

In [ ]:
data_filter <- data %>%
    mutate(discount_factor = 1 - tot_discounts/tot_charges) %>%
    mutate(price_num = (ip_charges + icu_charges + ancillary_charges)*discount_factor - tot_mcare_payment) %>%
    mutate(price_denom = tot_discharges - mcare_discharges) %>%
    mutate(price = price_num/price_denom) 

# Question 4
what is the distribution of estimated prices in each year? Present results with a violin plot (be sure to do something about outliers and/or negative prices in the data)


In [ ]:
data_filter4 <- data_filter %>%
    filter(year >= 2009 & year <= 2019) %>%
    mutate(price = tot_operating_exp / tot_discharges) %>%
    filter(is.finite(price), price > 1) %>%
    group_by(year) %>%
    mutate(
        Q1 = quantile(price, 0.25),
        Q3 = quantile(price, 0.75),
        IQR = Q3 - Q1,
        lower = Q1 - 1.5 * IQR,
        upper = Q3 + 1.5 * IQR
    ) %>%
    filter(price >= lower & price <= upper) %>%
    ungroup()

# violin plot
ggplot(data_filter4, aes(x = factor(year), y = price)) +
    geom_violin() +
    labs(
        title = "Distribution of estiamted prices",
        x = "fiscal year",
        y = "estimated price ($)"
    ) +
    theme_minimal()

# Question 5
What share of hospitals penalized under HRRP/VBP? Provide a graph showing the share of penalized hospitals from 2012-2019.


In [ ]:
penalty_summary <- data %>%
  filter(year >= 2012 & year <= 2019) %>%
  mutate(
    is_hrrp_pen = ifelse(!is.na(hrrp_payment) & hrrp_payment > 0, 1, 0),
    is_hvbp_pen = ifelse(!is.na(hvbp_payment) & hvbp_payment > 0, 1, 0),
    is_penalized = ifelse(is_hrrp_pen == 1 | is_hvbp_pen == 1, 1, 0)
  ) %>%
  group_by(year) %>%
  summarize(
    total_hospitals = n_distinct(provider_number),
    penalized_count = sum(is_penalized, na.rm = TRUE),
    share_penalized = penalized_count / total_hospitals
  )

# graph
ggplot(penalty_summary, aes(x = year, y = share_penalized)) +
  geom_line() +
  geom_point() +
  scale_y_continuous(labels = scales::label_percent()) + 
  scale_x_continuous(breaks = 2012:2019) +
  labs(
    title = "Share of Hospitals Penalized under HRRP/VBP",
    x = "Fiscal Year",
    y = "Share of Penalized Hospitals (%)"
  ) +
  theme_minimal()

# Question 6 
Provide a summary of OLS estimates of the effect of net penalties on price changes. Present your results in a table with three different specifications: 1) a “baseline” specification using only net penalty as a covariate; 2) “baseline” specification plus the pre-penalty (2009-2011) mean bed size; 3) “baseline” specification plus bed size plus pre-penalty (2009-2011) average Medicaid discharges.


In [ ]:
data_filter6 <- data_filter %>%
    filter(year >= 2009 & year <= 2014) %>%
    group_by(provider_number) %>%
    summarize(
        price_change = mean(price[year == 2014], na.rm = TRUE) - 
                       mean(price[year == 2011], na.rm = TRUE), 
        net_penalty = sum(c(hrrp_payment[year == 2012], 
                            hvbp_payment[year == 2012]), na.rm = TRUE),
        avg_mcare_discharge = mean(mcare_discharges[year <= 2011], na.rm = TRUE),
        avg_beds = mean(beds[year <= 2011], na.rm = TRUE),
        avg_mcaid_discharge = mean(mcaid_discharges[year <= 2011], na.rm = TRUE)
    ) %>% 
    ungroup() %>% 
    mutate(net_penalty = abs(pmin(net_penalty, 0))) %>% 
    filter(!is.na(price_change), !is.na(avg_mcare_discharge))

In [ ]:
model1 <- feols(price_change ~ net_penalty, data = data_filter6)
model2 <- feols(price_change ~ net_penalty + avg_beds, data = data_filter6)
model3 <- feols(price_change ~ net_penalty + avg_beds + avg_mcaid_discharge, data = data_filter6)

etable(model1, model2, model3, 
       title = "OLS Results",
       headers = c("Baseline", "with Bed Size", "with Medicaid"),
       tex = FALSE)

# Question 7
Provide a scatterplot of net penalty against pre-2012 Medicare discharges.


In [ ]:
scatterplot_data <- data_filter %>%
  filter(year >= 2009 & year <= 2012) %>%
  group_by(provider_number) %>%
  summarize(
    avg_mcare_discharges_09_11 = mean(mcare_discharges[year < 2012], na.rm = TRUE),
    net_penalty_2012 = abs(pmin(mean((hrrp_payment + hvbp_payment)[year == 2012], 
                                     na.rm = TRUE), 0)),
    .groups = "drop"
  ) %>%
  filter(!is.na(avg_mcare_discharges_09_11), !is.na(net_penalty_2012))

ggplot(scatterplot_data, aes(x = avg_mcare_discharges_09_11, y = net_penalty_2012)) +
  geom_point() + 
  geom_smooth(method = "lm", se = FALSE) + 
  scale_x_continuous(labels = scales::comma) + 
  scale_y_continuous(labels = scales::dollar) + 
  labs(
    title = "Net Penalty vs. Pre-2012 Medicare Discharges",
    x = "Average Medicare Discharges",
    y = "Average Net Penalty"
  ) +
  theme_minimal() 

# Question 8
Provide a summary of the first stage and reduced-form results using pre-penalty Medicare discharges as an instrument for net penalties. Present your results in a table with three different specifications as in Question 6.


In [ ]:
fs1 <- feols(net_penalty ~ avg_mcare_discharge, data = data_filter6)
fs2 <- feols(net_penalty ~ avg_mcare_discharge + avg_beds, data = data_filter6)
fs3 <- feols(net_penalty ~ avg_mcare_discharge + avg_beds + avg_mcaid_discharge, data = data_filter6)

rf1 <- feols(price_change ~ avg_mcare_discharge, data = data_filter6)
rf2 <- feols(price_change ~ avg_mcare_discharge + avg_beds, data = data_filter6)
rf3 <- feols(price_change ~ avg_mcare_discharge + avg_beds + avg_mcaid_discharge, data = data_filter6)

etable(fs1, fs2, fs3,
       title = "First Stafe Results",
       headers = c("Baseline", "with Bed Size", "with Medicaid"),
       tex = FALSE)

etable(rf1, rf2, rf3, 
       title = "Reduced Form Results",
       headers = c("Baseline", "with Bed Size", "with Medicaid"),
       tex = FALSE)

# Question 9
Provide a summary of IV estimates of the effect of net penalties on price changes. Again present your results in a table with the three different specifications as in Questions 6 and 8.


In [ ]:
iv1 <- feols(price_change ~ 1 | net_penalty ~ avg_mcare_discharge,
             data = data_filter6)
iv2 <- feols(price_change ~ avg_beds | net_penalty ~ avg_mcare_discharge,
             data = data_filter6)
iv3 <- feols(price_change ~ avg_beds + avg_mcaid_discharge | net_penalty ~ avg_mcare_discharge,
             data = data_filter6)

etable(iv1, iv2, iv3,
       title = "IV Estimates",
       headers = c("Baseline", "with Bed Size", "Medicaid"),
       fitstat = ~ r2 + n + wf,
       tex = FALSE)

# Question 10
Briefly explain the “Local” ATE in the context of your estimates. How might a local effect differ from an overall ATE in this setting?

The local ATE refers to the effect of the penalties on price changes for the hospitals whose penalty status was changed by the pre-2012 medicare discharges. The ATE would be the impact of penalties on any hospital chosen randomly; the local ATE is weighted towards larger Medicare-dependent hospitals.
